## **Register Silver Tables in Temp Views**

In [ ]:
# Lakehouse_Silver path — tables are under dbo schema
silver_path = "abfss://246f1c03-56af-4f6f-8640-aff9681c241b@onelake.dfs.fabric.microsoft.com/c2f0b5a0-0886-4239-ba4a-1129e098e73d/Tables/dbo"

silver_tables = ["silver_currencyexchange", "silver_customer", "silver_date",
                 "silver_orderrows", "silver_orders", "silver_product",
                 "silver_sales", "silver_store"]

for table in silver_tables:
    df = spark.read.format("delta").load(f"{silver_path}/{table}")
    df.createOrReplaceTempView(table)
    print(f"✅ Registered: {table}")

## **Explore the Product table**

In [ ]:
# Explore the Product Table
display(spark.sql("SELECT * FROM silver_product LIMIT 10"))

# Total Row Counts
spark.sql("SELECT COUNT(*) AS Total_Count FROM silver_product").show()

## **Gold: dim_product**

In [ ]:
dim_product = spark.sql("""
    SELECT
        ProductKey,
        ProductCode,
        ProductName,
        Manufacturer,
        Brand,
        Color,
        WeightUnit,
        Weight,
        Cost,
        Price,
        CategoryKey,
        CategoryName,
        SubCategoryKey,
        SubCategoryName
    FROM silver_product
""")

dim_product.write.format("delta").mode("overwrite").saveAsTable("dim_product")
print(f"✅ dim_product written — {dim_product.count()} rows")

## **Explore the Customer table**

In [ ]:
# Explore the Customer Table
display(spark.sql("SELECT * FROM silver_customer LIMIT 10"))

# Total Row Counts
spark.sql("SELECT COUNT(*) AS Total_Count FROM silver_customer").show()

## **Gold: dim_customer**

In [ ]:
dim_customer = spark.sql("""
    SELECT
        CustomerKey,
        GeoAreaKey,
        FullName,
        Gender,
        Title,
        City,
        State,
        StateFull,
        ZipCode,
        Country,
        CountryFull,
        Continent,
        Birthday,
        Age,
        Occupation,
        Company
    FROM silver_customer
""")

dim_customer.write.format("delta").mode("overwrite").saveAsTable("dim_customer")
print(f"✅ dim_customer written — {dim_customer.count()} rows")

## **Explore the Date table**

In [ ]:
# Explore the Date Table
display(spark.sql("SELECT * FROM silver_date LIMIT 10"))

# Total Row Count
spark.sql("SELECT COUNT(*) AS Total_Count FROM silver_date").show()

## **Gold: dim_date**

In [ ]:
dim_date = spark.sql("""
    SELECT
        DateKey,
        Date,
        Year,
        YearQuarter,
        YearQuarterNumber,
        Quarter,
        YearMonth,
        YearMonthShort,
        YearMonthNumber,
        Month,
        MonthShort,
        MonthNumber,
        DayofWeek,
        DayofWeekShort,
        DayofWeekNumber,
        WorkingDay,
        WorkingDayNumber
    FROM silver_date
""")

dim_date.write.format("delta").mode("overwrite").saveAsTable("dim_date")
print(f"✅ dim_date written — {dim_date.count()} rows")

## **Explore the Store table**

In [ ]:
# Explore the Store Table
display(spark.sql("SELECT * FROM silver_store LIMIT 10"))

# Total Row Count
spark.sql("SELECT COUNT(*) AS Total_Count FROM silver_store").show()

## **Gold: dim_store**

In [ ]:
dim_store = spark.sql("""
    SELECT
        StoreKey,
        StoreCode,
        GeoAreaKey,
        CountryCode,
        CountryName,
        State,
        OpenDate,
        CloseDate,
        Description,
        SquareMeters,
        Status
    FROM silver_store
""")

dim_store.write.format("delta").mode("overwrite").saveAsTable("dim_store")
print(f"✅ dim_store written — {dim_store.count()} rows")

## **Explore the Sales table**

In [ ]:
# Explore the Sales Table
display(spark.sql("SELECT * FROM silver_sales LIMIT 10"))

# Total Row Count
spark.sql("SELECT COUNT(*) AS Total_Count FROM silver_sales").show()

## **Gold: fact_sales**

In [ ]:
fact_sales = spark.sql("""
    SELECT
        s.OrderKey,
        s.LineNumber,
        s.OrderDate,
        s.DeliveryDate,
        s.CustomerKey,
        s.StoreKey,
        s.ProductKey,
        s.Quantity,
        s.UnitPrice,
        s.NetPrice,
        s.UnitCost,
        s.CurrencyCode,
        s.ExchangeRate,
        s.SalesAmount,
        s.GrossProfit,
        ROUND(s.SalesAmount / s.ExchangeRate, 2)   AS SalesAmountUSD,
        ROUND(s.GrossProfit / s.ExchangeRate, 2)   AS GrossProfitUSD
    FROM silver_sales s
    WHERE s.Quantity IS NOT NULL
      AND s.UnitPrice IS NOT NULL
      AND s.UnitCost IS NOT NULL
""")

fact_sales.write.format("delta").mode("overwrite").saveAsTable("fact_sales")
print(f"✅ fact_sales written — {fact_sales.count()} rows")